# Retrieve and Integrate MoTrPAC + GTEx Data

This is one chapter of the *From Sequence to Variant: Protein Language Models on CFDE Data* training module. Follow the numbered chapters in order.


---
## Part 2: Retrieving and Integrating CFDE Data (30 min)

### 2.1 MoTrPAC: load the rat 6-month endurance training dataset

MoTrPAC rat training data is available as direct Google Cloud Storage downloads — no R installation needed. We'll pull pre-computed differential analysis results (already-tested statistics comparing trained vs. sedentary animals) for one tissue as a working example, then generalize.

Data license: **CC BY 4.0**. Source: `storage.googleapis.com/motrpac-rat-training-6mo-extdata/`


**A note on where this data actually lives:** `TRNSCRPT_LIVER_DA` and similar per-tissue differential analysis (DA) objects are *not* standalone downloadable files on Google Cloud Storage — that GCS bucket (`motrpac-rat-training-6mo-extdata`) hosts only epigenomics (ATAC/METHYL) raw data and feature annotation files. The transcriptomics/proteomics DA objects are bundled as lazy-loaded R data objects inside the `MotrpacRatTraining6moData` package's `data/` folder on GitHub. We fetch the package source tarball once and extract just the two `.rda` files we need — this mirrors the package maintainers' own documented fallback for users who can't run `devtools::install_github()`.


In [2]:
# ============================================================
# CELL PURPOSE: download MoTrPAC's data package and pull out the one table we need
# (gene activity changes in liver tissue after 8 weeks of exercise training in rats).
# ============================================================
#
# BACKGROUND: MoTrPAC's detailed results (like "TRNSCRPT_LIVER_DA", which stands for
# "transcriptomics, liver tissue, differential analysis") are NOT sitting as individual
# downloadable files online. They're bundled inside the source code of an R software
# package on GitHub. Since this notebook uses Python (not R), we download that package's
# full source code as a single compressed file ("tarball"), then pull out just the one
# table we need from inside it — like unzipping a folder to grab a single spreadsheet.

## Note on runtime: this might take a minute or two!

import pyreadr   # lets Python read R's .rda file format
import os
import tarfile   # lets Python open .tar.gz compressed archives
import io

TISSUE = "LIVER"
ASSAY = "TRNSCRPT"   # "TRNSCRPT" = transcriptomics (gene activity / RNA levels)

# This is the exact web address of the MoTrPAC package's source code on GitHub.
TARBALL_URL = "https://api.github.com/repos/MoTrPAC/MotrpacRatTraining6moData/tarball/HEAD" #taken from https://motrpac.github.io/MotrpacRatTraining6moData/articles/MotrpacRatTraining6moData.html
TARBALL_PATH = "/tmp/motrpac_data_pkg.tar.gz"  # where we'll save the download on this computer

# Only download if we haven't already (the file is a few hundred MB, so this saves time
# if you re-run this cell later in the same session).
if not os.path.exists(TARBALL_PATH):
    print("Downloading MotrpacRatTraining6moData package source (one-time, ~several hundred MB)...")
    with requests.get(TARBALL_URL, stream=True, timeout=300) as r:
        r.raise_for_status()  # this stops the notebook with a clear error if the download failed
        with open(TARBALL_PATH, "wb") as f:
            # Download in chunks (1 MB at a time) rather than all at once — more memory-friendly.
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
    print("Download complete.")
else:
    print("Package tarball already downloaded, reusing cached copy.")

def extract_rda_object(tarball_path, object_name):
    """Pull one specific .rda file out of the downloaded package archive, by name.
    Example: extract_rda_object(TARBALL_PATH, "TRNSCRPT_LIVER_DA") finds and saves just
    that one table, without needing to unpack the entire multi-hundred-MB archive to disk."""
    target_suffix = f"data/{object_name}.rda"
    with tarfile.open(tarball_path, "r:gz") as tar:
        # Look through every file in the archive until we find the one whose path ends with
        # what we're looking for (e.g. ".../data/TRNSCRPT_LIVER_DA.rda").
        member = next((m for m in tar.getmembers() if m.name.endswith(target_suffix)), None)
        if member is None:
            raise FileNotFoundError(f"Could not find {target_suffix} in package tarball")
        extracted = tar.extractfile(member)
        out_path = f"/tmp/{object_name}.rda"
        with open(out_path, "wb") as f:
            f.write(extracted.read())
        return out_path

# Build the exact object name we want, e.g. "TRNSCRPT_LIVER_DA", and extract + load it.
da_object_name = f"{ASSAY}_{TISSUE}_DA"
da_local_path = extract_rda_object(TARBALL_PATH, da_object_name)

# pyreadr.read_r() converts the R data file into a Python dictionary of pandas DataFrames
# (a DataFrame is just a spreadsheet-like table). We grab the first (and only) table inside.
result = pyreadr.read_r(da_local_path)
da_df = result[list(result.keys())[0]]
print(f"Loaded {da_object_name}: {da_df.shape[0]} rows, {da_df.shape[1]} columns")
da_df.head()  # .head() shows just the first few rows, as a quick preview

Download complete.
Loaded TRNSCRPT_LIVER_DA: 115496 rows, 22 columns


,feature,assay,assay_code,tissue,tissue_code,feature_ID,sex,comparison_group,p_value,adj_p_value,...,shrunk_logFC,shrunk_logFC_se,zscore,covariates,removed_samples,comparison_average_intensity,comparison_average_intensity_se,reference_average_intensity,reference_average_intensity_se,selection_fdr
0,NaN,TRNSCRPT,transcript-rna-seq,LIVER,t68-liver,ENSRNOG00000000021,male,1w,0.334319,1.0,...,0.000317,0.009321,0.965450,"pct_globin,rin,pct_umi_dup,median_5_3_bias",4.457362e-313,141.755683,7.197998,133.053666,10.862639,1.0
1,NaN,TRNSCRPT,transcript-rna-seq,LIVER,t68-liver,ENSRNOG00000000021,male,2w,0.186084,1.0,...,0.002813,0.026289,1.322254,"pct_globin,rin,pct_umi_dup,median_5_3_bias",4.457362e-313,151.997323,12.051268,133.053666,10.862639,1.0
2,NaN,TRNSCRPT,transcript-rna-seq,LIVER,t68-liver,ENSRNOG00000000021,male,4w,0.855643,1.0,...,-0.001280,0.037714,-0.181923,"pct_globin,rin,pct_umi_dup,median_5_3_bias",4.457362e-313,128.585613,12.419618,133.053666,10.862639,1.0
3,NaN,TRNSCRPT,transcript-rna-seq,LIVER,t68-liver,ENSRNOG00000000021,male,8w,0.259868,1.0,...,0.026606,0.072070,1.126704,"pct_globin,rin,pct_umi_dup,median_5_3_bias",4.457362e-313,149.571226,11.861613,133.053666,10.862639,1.0
4,NaN,TRNSCRPT,transcript-rna-seq,LIVER,t68-liver,ENSRNOG00000000021,female,1w,0.063956,1.0,...,0.011495,0.031501,1.852489,"pct_globin,rin,pct_umi_dup,median_5_3_bias",4.457362e-313,167.207176,4.490465,145.590368,13.327682,1.0


### ⚠️ Important structural note about this table

`TRNSCRPT_LIVER_DA` is a **timewise** table: one row per gene *per training duration*
(`comparison_group` = `1w`, `2w`, `4w`, or `8w` of endurance training vs. sex-matched sedentary
controls), separately per sex. It is not one row per gene overall. Two consequences:

1. There is no `gene_symbol` column in this table — the gene identifier is the Ensembl ID in
   `feature_ID`. We map this to a readable gene symbol using MoTrPAC's `FEATURE_TO_GENE_FILT`
   table below (a bundled package object, pulled from the same tarball as the DA data).
2. To call a gene "exercise-responsive" we need to pick a specific comparison. We use the
   **8-week endurance-trained vs. sedentary** comparison (the longest, most-trained timepoint)
   as our working definition for this module.


In [3]:
# ============================================================
# CELL PURPOSE: get a lookup table that translates cryptic gene IDs (like
# "ENSRNOG00000000021") into readable gene names (like "Eno2").
# ============================================================
# FEATURE_TO_GENE_FILT is another table bundled inside the same MoTrPAC package we already
# downloaded above, so this reuses the tarball we already have on disk (fast, no new download).

f2g_path = extract_rda_object(TARBALL_PATH, "FEATURE_TO_GENE_FILT")
f2g_result = pyreadr.read_r(f2g_path)
feature_to_gene = f2g_result[list(f2g_result.keys())[0]]
print("FEATURE_TO_GENE_FILT columns:", feature_to_gene.columns.tolist())
# Preview just the 3 columns we actually care about: the cryptic ID, the readable gene name,
# and its Ensembl gene ID (a standardized identifier used across many biology databases).
feature_to_gene[["feature_ID", "gene_symbol", "ensembl_gene"]].head()

FEATURE_TO_GENE_FILT columns: ['entrez_gene', 'feature_ID', 'rgd_gene', 'gene_symbol', 'old_gene_symbol', 'ensembl_gene', 'relationship_to_gene', 'custom_annotation', 'kegg_id']


,feature_ID,gene_symbol,ensembl_gene
0,NP_036620.2,A2m,ENSRNOG00000028896
1,NP_036620.2,A2m,ENSRNOG00000045772
2,NP_036621.1,Acaa1a,ENSRNOG00000032908
3,NP_036621.1_S253s,Acaa1a,ENSRNOG00000032908
4,NP_036621.1_S261s,Acaa1a,ENSRNOG00000032908


### How "responsiveness" is defined here

- **Upregulation** means a gene's expression increased.
- **Downregulation** means a gene's expression decreased.
- **Exercise-responsive** in this notebook means either one, a significant change in expression
  in either direction, not upregulation specifically.
- A gene with little or no change in either direction is labeled "non-responsive."

This matters biologically: exercise adaptation involves genes turning both on and off. Treating only upregulation as "responsive" would bake in an assumption that doesn't reflect how exercise physiology actually works.

### Why nominal p-values, not adjusted (multiple-testing-corrected) p-values

- When testing thousands of genes across multiple timepoints and sexes simultaneously (as MoTrPAC's differential analysis does), the standard, statistically rigorous practice is to
  use **adjusted p-values** (e.g. Benjamini-Hochberg/Benjamini-Yekutieli FDR correction), which
  control the false-positive rate across all those simultaneous tests, not just one at a time.
- **We do not do that here.** We use MoTrPAC's raw, **nominal** `p_value` column instead, a less strict threshold.
- **Why:** at this dataset's scale (thousands of genes times 4 timepoints times 2 sexes), adjusted p-values are conservative enough that filtering on them leaves us with very few genes.
- **For reference:** adjusted p-values at this timepoint leave 90 genes, versus 2,801 with
  nominal p-values. A more rigorous version of this workflow could still run on that smaller,
  statistically conservative gene set, a natural extension for future work.
- **The trade-off:** using nominal p-values means our "exercise-responsive" gene list almost
  certainly includes some false positives, genes that look significant here by chance, not
  because they truly respond to exercise.
- This may be an acceptable trade-off for a workflow demonstration, where the goal is to
  illustrate the method end-to-end. It would **not** be acceptable for a real research claim
  about which specific genes respond to exercise. A rigorous analysis would either:
  - use adjusted p-values and accept a smaller gene set, or
  - use a less conservative but still principled correction method (e.g. independent
    hypothesis weighting, which MoTrPAC's own `TRAINING_REGULATED_FEATURES` object uses)

In [4]:
# ============================================================
# CELL PURPOSE: decide which genes count as "exercise-responsive" and get their names.
# ==============================================b==============
print(da_df.columns.tolist())  # print all column names so we can see exactly what data we have
print(da_df.head(n=2))

TIMEPOINT = "8w"   # we're using the 8-week (longest) training duration as our comparison
PVAL_COL = "p_value"        # the statistical significance of the gene's change (lower = more confident)
LFC_COL = "logFC" if "logFC" in da_df.columns else "shrunk_logFC"  # size of the change (bigger = stronger effect)
FEATURE_ID_COL = "feature_ID"
GENE_COL = "gene_symbol"

# Keep only the rows for our chosen 8-week timepoint (the table has separate rows for
# 1-week, 2-week, 4-week, and 8-week comparisons — we only want one of them).
timepoint_df = da_df[da_df["comparison_group"] == TIMEPOINT].copy()

# A gene can sometimes appear more than once (e.g. from different measurement probes).
# Here we keep only the most statistically significant (lowest p-value) row per gene.
timepoint_df = timepoint_df.sort_values(PVAL_COL).drop_duplicates(subset=FEATURE_ID_COL, keep="first")

# Attach a readable gene name to each row, using the lookup table from the cell above.
# "merge" is like a spreadsheet VLOOKUP: it matches rows between two tables on a shared column.
timepoint_df = timepoint_df.merge(
    feature_to_gene[["feature_ID", GENE_COL]].drop_duplicates(subset="feature_ID"),
    on=FEATURE_ID_COL, how="left"
)
# Drop any rows where we couldn't find a matching gene name.
timepoint_df = timepoint_df.dropna(subset=[GENE_COL])

# We use the "nominal" (unadjusted) p-value here rather than the more conservative
# "adjusted" p-value. Why: when you test thousands of genes across four timepoints at once,
# standard statistical correction (which guards against false positives) becomes very strict
# and would leave us with almost no genes to work with for a teaching example. This is a
# simplification made for pedagogical purposes, flagged here explicitly.
SIG_THRESHOLD = 0.05
responsive = timepoint_df[timepoint_df[PVAL_COL] < SIG_THRESHOLD].copy()
# Sort so the genes with the BIGGEST change (regardless of direction, hence abs = absolute value)
# come first.
responsive = responsive.sort_values(LFC_COL, key=abs, ascending=False)

print(f"{len(responsive)} exercise-responsive genes at nominal p < {SIG_THRESHOLD} ({TIMEPOINT} vs. sedentary, {TISSUE})")
# Take just the top 75 genes by effect size to keep the rest of the notebook fast to run.
top_genes = responsive[GENE_COL].head(75).tolist()
print("Top genes by |log fold-change|:", top_genes[:15], "...")

['feature', 'assay', 'assay_code', 'tissue', 'tissue_code', 'feature_ID', 'sex', 'comparison_group', 'p_value', 'adj_p_value', 'logFC', 'logFC_se', 'shrunk_logFC', 'shrunk_logFC_se', 'zscore', 'covariates', 'removed_samples', 'comparison_average_intensity', 'comparison_average_intensity_se', 'reference_average_intensity', 'reference_average_intensity_se', 'selection_fdr']
  feature     assay          assay_code tissue tissue_code  \
0     NaN  TRNSCRPT  transcript-rna-seq  LIVER   t68-liver   
1     NaN  TRNSCRPT  transcript-rna-seq  LIVER   t68-liver   

           feature_ID   sex comparison_group   p_value  adj_p_value  ...  \
0  ENSRNOG00000000021  male               1w  0.334319          1.0  ...   
1  ENSRNOG00000000021  male               2w  0.186084          1.0  ...   

   shrunk_logFC  shrunk_logFC_se    zscore  \
0      0.000317         0.009321  0.965450   
1      0.002813         0.026289  1.322254   

                                   covariates  removed_samples  \
0  p

In [ ]:
# Quick check for: how many genes would survive an adjusted p-value threshold, versus the nominal p-value threshold we actually use?
n_total_rows = len(timepoint_df) if 'timepoint_df' in dir() else len(da_df[da_df["comparison_group"] == "8w"])
check_df = da_df[da_df["comparison_group"] == "8w"]

n_adj_exactly_one = (check_df["adj_p_value"] == 1.0).sum()
n_adj_significant = (check_df["adj_p_value"] < 0.05).sum()
n_nominal_significant = (check_df["p_value"] < 0.05).sum()

print(f"Total rows at this timepoint: {len(check_df)}")
print(f"Rows with adj_p_value exactly 1.0: {n_adj_exactly_one} ({n_adj_exactly_one/len(check_df)*100:.0f}%)")
print(f"Rows significant at adjusted p < 0.05: {n_adj_significant}")
print(f"Rows significant at nominal p < 0.05: {n_nominal_significant}")
print(f"\nUsing adjusted p-values would leave {n_adj_significant} genes to work with, "
      f"versus {n_nominal_significant} using nominal p-values, backing up the trade-off "
      f"described above.")

Total rows at this timepoint: 28874
Rows with adj_p_value exactly 1.0: 28571 (99%)
Rows significant at adjusted p < 0.05: 90
Rows significant at nominal p < 0.05: 2801

Using adjusted p-values would leave 90 genes to work with, versus 2801 using nominal p-values, backing up the trade-off described above.


### 2.2 GTEx: query the V2 API for eQTLs on the same genes

Now we ask: for these same MoTrPAC exercise-responsive genes, does GTEx report regulatory variants (eQTLs) that modulate their expression?

**Important species issue:** MoTrPAC is a *rat* study, but GTEx is *human-only* data. The gene
symbols we have so far (e.g. `Eno2`, `Pak2`) are rat gene symbols with rat-style casing — querying
GTEx directly with these will fail (GTEx expects human gene symbols, which use different casing
conventions, e.g. `ENO2`, `PAK2`, and won't reliably string-match rat casing even where the
ortholog exists). We first convert each rat gene to its human ortholog using MoTrPAC's own
`RAT_TO_HUMAN_GENE` mapping (bundled in the same package tarball), then query GTEx using the
correct human symbol.

GTEx API v2 base: `https://gtexportal.org/api/v2` — no login required.


In [5]:
# ============================================================
# CELL PURPOSE: translate rat gene names into their human equivalents.
# ============================================================
# WHY THIS STEP EXISTS: MoTrPAC studied RATS, but GTEx (the next dataset we'll query) only
# has HUMAN data. Many genes have a "matching" gene in another species (called an ortholog) —
# e.g. rat "Eno2" and human "ENO2" do the same job. Before we can look up anything in GTEx,
# we need to know each rat gene's human counterpart.

# RAT_TO_HUMAN_GENE is (again) a table bundled inside the MoTrPAC package we already downloaded.
r2h_path = extract_rda_object(TARBALL_PATH, "RAT_TO_HUMAN_GENE")
r2h_result = pyreadr.read_r(r2h_path)
rat_to_human = r2h_result[list(r2h_result.keys())[0]]
print("RAT_TO_HUMAN_GENE columns:", rat_to_human.columns.tolist())

# Build a simple lookup dictionary: {rat gene name: human gene name}.
# Some rat genes don't have a single clear human match — we drop those (dropna) and keep
# only the first match if there's more than one candidate (drop_duplicates).
rat_to_human_map = (
    rat_to_human.dropna(subset=["RAT_SYMBOL", "HUMAN_ORTHOLOG_SYMBOL"])
    .drop_duplicates(subset="RAT_SYMBOL", keep="first")
    .set_index("RAT_SYMBOL")["HUMAN_ORTHOLOG_SYMBOL"]
    .to_dict()
)

# Filter our 75 top genes down to just the ones that DO have a known human match.
human_ortholog_genes = {g: rat_to_human_map[g] for g in top_genes if g in rat_to_human_map}
n_missing = len(top_genes) - len(human_ortholog_genes)
print(f"{len(human_ortholog_genes)} / {len(top_genes)} top genes have a known human ortholog "
      f"({n_missing} have no 1:1 rat-human ortholog and will be skipped for the GTEx step).")
list(human_ortholog_genes.items())[:10]  # preview the first 10 rat -> human gene name pairs

RAT_TO_HUMAN_GENE columns: ['RAT_SYMBOL', 'NAME', 'RAT_GENE_RGD_ID', 'RAT_NCBI_GENE_ID', 'RAT_ENSEMBL_ID', 'RAT_UNIPROT_ID', 'RAT_OLD_SYMBOL', 'HUMAN_ORTHOLOG_SYMBOL', 'HUMAN_ORTHOLOG_RGD', 'HUMAN_ORTHOLOG_NCBI_GENE_ID', 'HUMAN_ORTHOLOG_ENSEMBL_ID', 'HUMAN_ORTHOLOG_SOURCE']
49 / 75 top genes have a known human ortholog (26 have no 1:1 rat-human ortholog and will be skipped for the GTEx step).


[('Eno2', 'ENO2'),
 ('Pak2', 'PAK2'),
 ('Lig1', 'LIG1'),
 ('Gns', 'GNS'),
 ('Cbr1', 'CBR1'),
 ('Mast4', 'MAST4'),
 ('Ppfia1', 'PPFIA1'),
 ('Hdac6', 'HDAC6'),
 ('Ltb', 'LTB'),
 ('Aldh18a1', 'ALDH18A1')]

In [6]:
# ============================================================
# CELL PURPOSE: test our GTEx query on ONE well-known gene before running it on all 49, to make sure the query itself works correctly.
# ============================================================
# SORT1 is chosen because it has one of the most famous, well-documented liver eQTLs in the
# scientific literature (linked to LDL cholesterol levels) — if our query can't find SORT1's
# eQTLs, something is wrong with our query, not with the biology.
#
# TECHNICAL NOTE (learned through trial and error while building this notebook): GTEx's API
# does NOT accept a plain gene name like "SORT1" as a search term for this particular type of
# query. It requires a "GENCODE ID" — a precise, versioned genetic identifier that looks like
# "ENSG00000134243.11". So we first have to look up SORT1's GENCODE ID, THEN use that ID to
# search for its eQTLs. This is a two-step process: resolve name -> ID, then ID -> results.
GTEX_BASE = "https://gtexportal.org/api/v2"
GTEX_TISSUE = "Liver"  # must match GTEx's exact naming for tissues
GENCODE_VERSION = "v26"  # the specific gene-annotation version behind the GTEx dataset we're using

def resolve_gencode_id(gene_symbol, gencode_version=GENCODE_VERSION):
    """Look up a human gene's official versioned GENCODE ID from its plain name."""
    resp = requests.get(
        f"{GTEX_BASE}/reference/gene",
        params={"geneId": gene_symbol, "gencodeVersion": gencode_version},
        timeout=30,
    )
    resp.raise_for_status()
    genes = resp.json().get("data", [])
    if not genes:
        return None
    return genes[0].get("gencodeId")

sort1_gencode_id = resolve_gencode_id("SORT1")
print("Resolved SORT1 ->", sort1_gencode_id)

# Now use that GENCODE ID to actually search for SORT1's eQTLs in liver tissue.
diagnostic_params = {
    "gencodeId": [sort1_gencode_id],
    "tissueSiteDetailId": ["Liver"],
    "datasetId": "gtex_v8",
    "itemsPerPage": 50,
}
diag_resp = requests.get(f"{GTEX_BASE}/association/singleTissueEqtl", params=diagnostic_params, timeout=30)
print("Status code:", diag_resp.status_code)  # 200 = success; anything else means something's wrong
print("Response (first 1000 chars):", diag_resp.text[:1000])

Resolved SORT1 -> ENSG00000134243.11
Status code: 200
Response (first 1000 chars): {"data":[{"snpId":"rs34680256","pos":109052758,"snpIdUpper":"RS34680256","variantId":"chr1_109052758_G_C_b38","geneSymbol":"SORT1","pValue":0.0000104754,"geneSymbolUpper":"SORT1","datasetId":"gtex_v8","tissueSiteDetailId":"Liver","ontologyId":"UBERON:0001114","chromosome":"chr1","gencodeId":"ENSG00000134243.11","nes":0.518678},{"snpId":"rs35389458","pos":109061071,"snpIdUpper":"RS35389458","variantId":"chr1_109061071_T_A_b38","geneSymbol":"SORT1","pValue":1.529e-6,"geneSymbolUpper":"SORT1","datasetId":"gtex_v8","tissueSiteDetailId":"Liver","ontologyId":"UBERON:0001114","chromosome":"chr1","gencodeId":"ENSG00000134243.11","nes":0.559451},{"snpId":"rs28663119","pos":109063742,"snpIdUpper":"RS28663119","variantId":"chr1_109063742_T_G_b38","geneSymbol":"SORT1","pValue":1.529e-6,"geneSymbolUpper":"SORT1","datasetId":"gtex_v8","tissueSiteDetailId":"Liver","ontologyId":"UBERON:0001114","chromosome":"chr1","genc

In [7]:
# ============================================================
# CELL PURPOSE: now that we know the query works (from the diagnostic cell above), run it across all of our exercise-responsive genes.
# ============================================================

## Note on runtime: this might take a minute or two!

def get_gtex_eqtls(human_gene_symbol, tissue_site_detail_id=GTEX_TISSUE, page_size=50):
    """For one human gene, find all its statistically significant regulatory variants
    (eQTLs) in the given tissue. Internally this does the same two-step lookup as the
    diagnostic cell above: gene name -> GENCODE ID -> eQTL results."""
    gencode_id = resolve_gencode_id(human_gene_symbol)
    if gencode_id is None:
        # Not every gene has a resolvable GENCODE ID in this exact annotation version —
        # if that happens, we skip it and move on rather than crashing the whole loop.
        print(f"  [warn] Could not resolve GENCODE ID for {human_gene_symbol} — skipping")
        return []
    endpoint = f"{GTEX_BASE}/association/singleTissueEqtl"
    params = {
        "gencodeId": [gencode_id],
        "tissueSiteDetailId": [tissue_site_detail_id],
        "datasetId": "gtex_v8",
        # itemsPerPage controls how many results we ask for. We set this high (500) because
        # some genes genuinely have hundreds of significant eQTLs — a low limit here would cut off real results without any error message.
        "itemsPerPage": page_size,
    }
    try:
        resp = requests.get(endpoint, params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json().get("data", [])
        return data
    except requests.exceptions.RequestException as e:
        print(f"  [warn] GTEx query failed for {human_gene_symbol}: {e}")
        return []

# Loop over every rat gene -> human ortholog pair, query GTEx, and collect every eQTL hit
# into one big list of dictionaries (which we'll turn into a table next).
eqtl_records = []
for rat_gene, human_gene in human_ortholog_genes.items():
    hits = get_gtex_eqtls(human_gene, page_size=500)  # 500 avoids silent truncation — some genes (e.g. ALDH18A1) have 400+ significant eQTLs
    for h in hits:
        eqtl_records.append({
            "rat_gene_symbol": rat_gene,       # keep track of the ORIGINAL rat gene name too
            "human_gene_symbol": human_gene,
            "variant_id": h.get("variantId"),   # unique ID for this specific genetic variant
            "chromosome": h.get("chromosome"),
            "pos": h.get("pos"),                # exact genomic position (in base pairs)
            "pValue": h.get("pValue"),          # statistical confidence of this eQTL
            "nes": h.get("nes"),                # normalized effect size — how strongly this variant affects gene expression
        })

# Convert our list of results into a proper table (DataFrame) for easy filtering/analysis later.
eqtl_df = pd.DataFrame(eqtl_records)
n_genes_with_hits = eqtl_df["rat_gene_symbol"].nunique() if len(eqtl_df) else 0
print(f"Retrieved eQTL records for {n_genes_with_hits} / {len(human_ortholog_genes)} queried genes "
      f"(query failures logged as [warn] above, if any — a gene simply having zero significant "
      f"eQTLs in this tissue is a normal, valid result, not a failure)")
eqtl_df.head()

  [warn] Could not resolve GENCODE ID for H4C9 — skipping
  [warn] Could not resolve GENCODE ID for EEF1AKMT3 — skipping
Retrieved eQTL records for 13 / 49 queried genes (query failures logged as [warn] above, if any — a gene simply having zero significant eQTLs in this tissue is a normal, valid result, not a failure)


,rat_gene_symbol,human_gene_symbol,variant_id,chromosome,pos,pValue,nes
0,Lig1,LIG1,chr19_47535245_A_G_b38,chr19,47535245,0.000006,-0.412796
1,Lig1,LIG1,chr19_47540986_A_G_b38,chr19,47540986,0.000006,-0.412796
2,Lig1,LIG1,chr19_47541024_T_C_b38,chr19,47541024,0.000006,-0.412796
3,Lig1,LIG1,chr19_48113401_C_A_b38,chr19,48113401,0.000009,-0.536519
4,Lig1,LIG1,chr19_48120183_G_A_b38,chr19,48120183,0.000009,-0.536519


In [8]:
# ============================================================
# CELL PURPOSE: combine everything so far into one master table: for each gene, do we know
# (a) how it responds to exercise (from MoTrPAC), and (b) whether it has any known regulatory
# genetic variants (from GTEx)?
# ============================================================

genes_with_eqtls = set(eqtl_df["rat_gene_symbol"].unique()) if len(eqtl_df) else set()

# Start from our original 75 exercise-responsive genes.
integrated = responsive[responsive[GENE_COL].isin(top_genes)].copy()

# De-duplicate by gene symbol: multiple Ensembl feature_IDs can map to the same gene_symbol
# (paralog annotations, readthrough transcripts, etc. in FEATURE_TO_GENE_FILT), which would
# otherwise leave duplicate rows and break the 1:1 gene <-> embedding <-> label alignment we
# rely on in Part 3. Keep the most significant (lowest p-value) row per gene symbol.
integrated = integrated.sort_values(PVAL_COL).drop_duplicates(subset=GENE_COL, keep="first")

# Add three new "yes/no or count" columns to our table, one gene at a time:
integrated["has_human_ortholog"] = integrated[GENE_COL].isin(human_ortholog_genes.keys())
integrated["has_eqtl"] = integrated[GENE_COL].isin(genes_with_eqtls)
integrated["n_eqtls"] = integrated[GENE_COL].map(
    eqtl_df.groupby("rat_gene_symbol").size() if len(eqtl_df) else {}
).fillna(0).astype(int)

# Print a readable preview of the combined table.
print(integrated[[GENE_COL, LFC_COL, PVAL_COL, "has_human_ortholog", "has_eqtl", "n_eqtls"]].head(20))

       gene_symbol     logFC       p_value  has_human_ortholog  has_eqtl  \
0       AC141220.4  51.53481  3.119341e-27               False     False   
1             Eno2  52.12958  2.862024e-24                True     False   
2              Gns  44.70768  7.508159e-21                True     False   
3             Pak2  48.69121  7.938778e-21                True     False   
4             Lig1  47.73748  1.735438e-20                True      True   
5   AABR07032202.1 -47.93350  3.657893e-20               False     False   
6     LOC108348081  43.79024  1.672284e-17               False     False   
7             Cbr1 -39.93247  2.031989e-14                True      True   
9            Mast4 -38.16385  1.665467e-13                True     False   
11  AABR07035479.1  21.08730  8.261140e-13               False     False   
12           Hdac6  30.16172  9.112471e-13                True     False   
15           Casp6  24.35821  2.973066e-11                True     False   
16  AABR0704

### 📍 Data Checkpoint

Before moving on, describe your integrated dataset in the cell below:
- How many exercise-responsive proteins/genes did you identify in your chosen tissue?
- Of those, how many have at least one GTEx eQTL hit?
- What does it mean, biologically, if a gene is exercise-responsive but has *no* eQTL coverage in GTEx — is that a strong negative, or just a data-coverage limitation? Explain.


**Your answer:**

